In [45]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
import sys
from pathlib import Path
from openai import OpenAI


import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))
from WebScrapper.scrape_site import scrape_site


In [46]:
load_dotenv(override=True)
api_key = os.getenv("GEMMA_API_KEY")

if api_key and len(api_key) > 10:
    print("API Key loaded successfully.")
else:
    print("Failed to load API Key. Please check your .env file.")

MODEL = "gemma3:latest"
GEMMA_BASE_URL = os.getenv("GEMMA_BASE_URL")

gemma = OpenAI(base_url=GEMMA_BASE_URL, api_key=api_key)

API Key loaded successfully.


In [47]:
link_system_prompt = """
You are provided with a list of website links and the contents of those web pages. 
You are able to decide which links are relevant and must include in the creative brochure, such as About page or Company Page etc.

You should respond in JSON as in this example:

{
  "start_url": "https://www.mcdonalds.com/ca/en-ca.html",
  "pages_scraped": 15,
  "pages": [
    {
      "title": "Title of the page",
      "text": "Contents of the page",
      "url": "https://url.com"
    },

"""

In [48]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links and contents on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email and Discord links.

Links (some might be relative links):

"""
    links = scrape_site(url)
    #print(links)
    user_prompt += "\n".join(links)
    return user_prompt

In [50]:
def select_relevant_links(url):
    response = gemma.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )

    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [56]:
select_relevant_links("https://www.apple.com")

{'start_url': 'https://www.apple.com/',
 'pages_scraped': 35,
 'pages': [{'title': 'Apple - Our Products',
   'text': "Discover Apple's innovative products, including iPhone, iPad, Mac, Apple Watch, AirPods, and Apple TV. Experience the seamless integration of hardware and software powered by the Apple ecosystem.",
   'url': 'https://www.apple.com/ca/shop/products'},
  {'title': 'Apple - Our Services',
   'text': 'Explore Apple Services, including Apple Music, Apple TV+, Apple Arcade, iCloud, AppleCare, and more. Enhance your digital life with a suite of services designed for seamless connectivity and entertainment.',
   'url': 'https://www.apple.com/ca/shop/services'},
  {'title': 'Apple - About Apple',
   'text': 'Learn about Apple’s history, mission, values, and commitment to innovation. Discover how Apple creates products and services that empower people around the world.',
   'url': 'https://www.apple.com/ca/about-apple/'},
  {'title': 'Apple - Newsroom',
   'text': 'Stay up-to-da

In [57]:
brochure_system_prompt = """
You are a sales expert and copywriter. Your task is to create a compelling and engaging brochure for a company based on the provided web page contents.
you always generate creative and catchy brochure content that atracts customers. Creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [58]:
def format_scraped_content(scraped_data: dict) -> str:
    output = []

    for page in scraped_data.get("pages", []):
        title = page.get("title", "")
        url = page.get("url", "")
        text = page.get("text", "")

        output.append(f"## {title}\nURL: {url}\n{text}\n")

    return "\n".join(output)


In [59]:
def get_brochure_user_prompt(company_name, url):
    scraped_data = select_relevant_links(url)  # returns dict
    content_text = format_scraped_content(scraped_data)  # convert to str

    user_prompt = f"""You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages.
Use this information to build a short brochure of the company in markdown without code blocks.

{content_text}
"""
    return user_prompt


In [60]:
get_brochure_user_prompt("Apple", "https://www.apple.com")

'You are looking at a company called: Apple\nHere are the contents of its landing page and other relevant pages.\nUse this information to build a short brochure of the company in markdown without code blocks.\n\n\n'

In [61]:
def create_brochure(company_name, url):
    response = gemma.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [39]:
create_brochure("Apple", "https://www.apple.com/in/")

**Apple: Imagine Beyond**

At Apple, we believe in the power of technology to transform lives. We craft experiences – from the intuitive intelligence of our iPhones to the boundless creativity unlocked by our Macs and iPads – designed to inspire and empower. 

**A World of Possibilities**

Explore a range of products engineered for exceptional performance and designed with unparalleled attention to detail. Experience the future of mobile with the iPhone, a device that seamlessly connects you to what matters most. Unleash your creativity with the powerful and versatile Mac lineup, including the iMac and MacBook. Discover the perfect tablet with iPad, designed for work, play, and everything in between. 

**Innovation. Driven by You.**

We are relentlessly committed to innovation, pushing the boundaries of what’s possible and constantly evolving to meet your needs.  Featuring the latest macOS Ventura and iPadOS 16, our platforms deliver unmatched productivity, entertainment, and a richer digital experience. 

**More Than Products. A Community.**

Join a global community of innovators, creators, and thinkers.  Visit an Apple Store to discover products firsthand and connect with our team.  Access dedicated support through Apple Support, and enjoy peace of mind with AppleCare+.

**Careers at Apple**

Shape the future with us. Apple offers dynamic career opportunities across a wide range of fields.  Explore our opportunities at [website address - omitted].

**Protecting What Matters.**

Your privacy and security are paramount. Apple is dedicated to safeguarding your personal information with advanced security features and a commitment to responsible data practices.

**A Sustainable Future.**

We’re building a better future for all. Apple is committed to reducing our environmental impact and creating a world where technology and nature thrive.

**Apple.  It’s personal.**